# 02 CCPD to YOLO: visual check (M1b-2)
Runs on Colab (Drive mounted). Builds `ccpd_yolo.tar` with `python -m alpr.ccpd yolo`, extracts it on the VM, checks counts and draws the converted boxes on a seeded sample.
**This notebook shows CCPD images** (the CCPD repo terms are unchecked, see PLAN.md). The repo is public: strip the image outputs before committing (see the last cell).

In [1]:
import os, random, shutil, tarfile
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageDraw
from google.colab import drive

In [2]:
# Parameters
SEED = 0
DRIVE = '/content/drive/MyDrive/alpr'
ARCHIVES = f'{DRIVE}/archives'
SAMPLE_TAR = f'{ARCHIVES}/ccpd_sample.tar'  # input, built in M1a
SPLIT_CSV = f'{ARCHIVES}/ccpd_split.csv'    # input, M1b-1
YOLO_TAR = f'{ARCHIVES}/ccpd_yolo.tar'      # output, kept on Drive
YOLO_TAR_LOCAL = '/content/ccpd_yolo.tar'   # VM copy
YOLO_DIR = '/content/ccpd_yolo'             # extracted on the VM
REPORTS = f'{DRIVE}/reports/ccpd_yolo'      # the grid PNG goes here
N_SAMPLE = 6   # random images per split in the grid
N_EXTREME = 4  # smallest and largest boxes (keep <= N_SAMPLE)

## Setup

In [3]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
%%bash
set -e
cd /content
if [ -d vehicle-plate-recognition ]; then git -C vehicle-plate-recognition pull --ff-only; else git clone https://github.com/medrees-1000/vehicle-plate-recognition.git; fi
pip install -q uv  # approved raw pip: only to bootstrap uv on Colab
uv pip install --system -q --no-deps -e /content/vehicle-plate-recognition

Already up to date.


In [5]:
# Not in the top imports cell on purpose: the package only exists after the install cell above.
from alpr import ccpd  # ImportError on the first run? Restart the kernel once: an editable install is not visible to an already-running kernel.
print('alpr.ccpd:', ccpd.__file__)
for p in (SAMPLE_TAR, SPLIT_CSV):
    assert os.path.exists(p), f'missing on Drive: {p}'

alpr.ccpd: /content/vehicle-plate-recognition/src/alpr/ccpd.py


## Step 3: convert
Reads the sample tar and the split file, checks every image and writes `ccpd_yolo.tar` plus `ccpd_yolo.dropped.csv` to Drive. If the tar already exists the command prints "skipping" and no counts; the dropped-per-reason table below is read from the CSV, so it shows either way. To see the command's own printout again, delete the tar and the CSV first.

In [6]:
assert os.path.exists(YOLO_TAR), 'ccpd_yolo.tar missing on Drive: refusing to rebuild it. To build it on purpose, comment out this line.'
!python -m alpr.ccpd yolo "{SAMPLE_TAR}" "{SPLIT_CSV}" "{YOLO_TAR}"
dropped = pd.read_csv(Path(YOLO_TAR).with_suffix('.dropped.csv'))
print('dropped per (reason, split):')
print(dropped.groupby(['reason', 'split']).size() if len(dropped) else 'nothing dropped')

/content/drive/MyDrive/alpr/archives/ccpd_yolo.tar exists, skipping
dropped per (reason, split):
reason   split
box_gap  train    44
         val       3
dtype: int64


## Step 4: copy to the VM disk and extract
Training reads from local disk, never from the Drive mount. The VM copy must have the same MD5 as the Drive file.

In [7]:
if not os.path.exists(YOLO_TAR_LOCAL):
    shutil.copyfile(YOLO_TAR, YOLO_TAR_LOCAL)
md5 = ccpd.file_md5(YOLO_TAR_LOCAL)
assert md5 == ccpd.file_md5(YOLO_TAR), 'VM copy differs from the Drive file'
print('ccpd_yolo.tar MD5 (record in PLAN.md):', md5)
if not Path(YOLO_DIR, 'data.yaml').exists():
    with tarfile.open(YOLO_TAR_LOCAL) as t:
        t.extractall(YOLO_DIR, filter='data')
print('extracted to', YOLO_DIR)

ccpd_yolo.tar MD5 (record in PLAN.md): bd44986acd6691a4f90fbea1e6017682
extracted to /content/ccpd_yolo


## Step 5: data.yaml and counts
Expected = rows in `ccpd_split.csv` minus the dropped images, per split. The assert fails if the extracted files do not match.

In [8]:
root = Path(YOLO_DIR)
print(root.joinpath('data.yaml').read_text())
split = pd.read_csv(SPLIT_CSV)
split['base'] = split['member'].str.rsplit('/', n=1).str[-1]
want = split.groupby('split').size().sub(dropped.groupby('split').size(), fill_value=0).astype(int)
rows, names = [], set()
for s in ('train', 'val', 'heldout'):
    imgs = sorted(root.joinpath('images', s).glob('*.jpg'))
    labs = sorted(root.joinpath('labels', s).glob('*.txt'))
    assert [p.stem for p in imgs] == [p.stem for p in labs], f'{s}: images and labels differ'
    names |= {p.name for p in imgs}
    rows.append((s, len(imgs), len(labs), int(want[s])))
table = pd.DataFrame(rows, columns=['split', 'images', 'labels', 'expected'])
print(table.to_string(index=False))
assert (table['images'] == table['expected']).all(), 'counts differ from split csv minus dropped'
bad = 0
for p in root.glob('labels/*/*.txt'):
    f = p.read_text().split()
    bad += not (len(f) == 5 and f[0] == '0' and all(0 < float(v) <= 1 for v in f[1:]))
print('label files not in the form `0 cx cy w h` within (0, 1]:', bad)
assert bad == 0
print('kept per label and split (from the extracted files):')
kept = split[split['base'].isin(names)]
print(pd.crosstab(kept['subset'], kept['split'], margins=True))

train: images/train
val: images/val
test: images/heldout
names:
  0: plate

  split  images  labels  expected
  train   16556   16556     16556
    val     997     997       997
heldout     300     300       300
label files not in the form `0 cx cy w h` within (0, 1]: 0
kept per label and split (from the extracted files):
split           heldout  train  val    All
subset                                    
ccpd_base            38   2031  122   2191
ccpd_blur            38   2075  125   2238
ccpd_challenge       38   2075  125   2238
ccpd_db              38   2075  125   2238
ccpd_fn              37   2075  125   2237
ccpd_rotate          37   2075  125   2237
ccpd_tilt            37   2075  125   2237
ccpd_weather         37   2075  125   2237
All                 300  16556  997  17853


## Step 6: draw the converted boxes
Seeded random sample (train and held-out), then the smallest and largest boxes over all splits. Titles show no filenames or plate text.

In [ ]:
def read_box(img: Path) -> tuple[float, float, float, float]:
    _, cx, cy, w, h = root.joinpath('labels', img.parent.name, img.stem + '.txt').read_text().split()
    return float(cx), float(cy), float(w), float(h)

def draw(img: Path) -> Image.Image:
    cx, cy, w, h = read_box(img)
    im = Image.open(img).convert('RGB')
    W, H = im.size
    ImageDraw.Draw(im).rectangle([(cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H], outline='red', width=5)
    return im

by_split = {s: sorted(root.joinpath('images', s).glob('*.jpg')) for s in ('train', 'val', 'heldout')}
by_area = sorted(((read_box(p)[2] * read_box(p)[3], p) for ps in by_split.values() for p in ps), key=lambda t: t[0])
rng = random.Random(SEED)
panels = [
    [(p, 'train') for p in rng.sample(by_split['train'], N_SAMPLE)],
    [(p, 'held-out') for p in rng.sample(by_split['heldout'], N_SAMPLE)],
    [(p, f'smallest: {a:.2%} of image') for a, p in by_area[:N_EXTREME]],
    [(p, f'largest: {a:.2%} of image') for a, p in by_area[-N_EXTREME:]],
]
fig, axes = plt.subplots(len(panels), N_SAMPLE, figsize=(3.2 * N_SAMPLE, 2.6 * len(panels)))
for ax in axes.ravel():
    ax.axis('off')
for row, items in zip(axes, panels):
    for ax, (p, title) in zip(row, items):
        ax.imshow(draw(p))
        ax.set_title(title, fontsize=9)
fig.suptitle('CCPD converted YOLO boxes (red)')
fig.tight_layout()

## Step 6b: why do the largest boxes look shifted?
Converts the YOLO label of the 4 largest boxes back to pixels and compares it with (a) the box in the CCPD filename and (b) the rectangle around the 4 vertices. If YOLO minus (a) is about 0, our converter and plotting are fine and any shift is in CCPD's own labels. Then the same numbers for the whole kept dataset, and a zoomed plot of the 4 images. Output is numbers only.

In [10]:
member_of = dict(zip(split['base'], split['member']))  # basename -> archive member path (ccpd_split.csv)
top = [p for _, p in by_area[-N_EXTREME:]]  # the largest boxes, same images as the grid

def boxes_px(img: Path) -> tuple[list[float], list[float], list[float]]:
    """Pixel boxes [x1, y1, x2, y2]: YOLO label converted back, filename box, rectangle around the 4 vertices."""
    cx, cy, w, h = read_box(img)
    with Image.open(img) as im:
        W, H = im.size
    lab = ccpd.parse_name(member_of[img.name])
    xs, ys = [q[0] for q in lab.vertices], [q[1] for q in lab.vertices]
    return [(cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H], list(lab.bbox), [min(xs), min(ys), max(xs), max(ys)]

def diff(a: list[float], b: list[float]) -> list[float]:
    return [p - q for p, q in zip(a, b)]  # signed, per edge x1 y1 x2 y2

def mean_xy(d: list[float]) -> tuple[float, float]:
    return (d[0] + d[2]) / 2, (d[1] + d[3]) / 2

def gap_pct(f: list[float], v: list[float]) -> float:
    return max(abs(d) for d in diff(f, v)) / max(f[2] - f[0], f[3] - f[1]) * 100  # same definition as the box_gap filter

print('4 largest boxes. Offsets in px = YOLO box minus reference; edges are x1 y1 x2 y2; no filenames.')
means = {'filename': [], 'vertex rect': []}
for i, p in enumerate(top, 1):
    y, f, v = boxes_px(p)
    print(f'image {i}: box is {read_box(p)[2] * read_box(p)[3]:.1%} of the image; filename box vs vertex rect gap {gap_pct(f, v):.1f}% of longer side')
    for name, ref in (('filename', f), ('vertex rect', v)):
        d = diff(y, ref)
        means[name].append(mean_xy(d))
        print(f'  YOLO - {name:11s} edges {[round(x, 2) for x in d]}  mean dx {mean_xy(d)[0]:+.2f}  dy {mean_xy(d)[1]:+.2f}')
for name, m in means.items():
    print(f'mean over the {len(m)} images, YOLO - {name}: dx {sum(a for a, _ in m) / len(m):+.2f}  dy {sum(b for _, b in m) / len(m):+.2f}')

rows = []
for a, p in by_area:  # every kept image in train, val and held-out
    y, f, v = boxes_px(p)
    rows.append((a, max(abs(d) for d in diff(y, f)), gap_pct(f, v)))
allb = pd.DataFrame(rows, columns=['area', 'yolo_vs_name_px', 'gap_pct'])
print(f'\nall {len(allb)} images: max |YOLO - filename box| over every edge = {allb.yolo_vs_name_px.max():.4f} px')
allb['bucket'] = pd.qcut(allb['area'], 3, labels=['small', 'medium', 'large'])  # thirds of the box area
g = allb.groupby('bucket', observed=True)
print('box area, % of image, per bucket:')
print((g['area'].agg(['min', 'max']) * 100).round(2))
print('gap between filename box and vertex rectangle, % of longer side (box_gap filter limit: 10):')
print(g['gap_pct'].agg(images='size', median='median', max='max').round(2))

4 largest boxes. Offsets in px = YOLO box minus reference; edges are x1 y1 x2 y2; no filenames.
image 1: box is 28.8% of the image; filename box vs vertex rect gap 0.0% of longer side
  YOLO - filename    edges [0.0, 0.0, -0.0, -0.0]  mean dx +0.00  dy +0.00
  YOLO - vertex rect edges [0.0, 0.0, -0.0, -0.0]  mean dx +0.00  dy +0.00
image 2: box is 29.7% of the image; filename box vs vertex rect gap 0.0% of longer side
  YOLO - filename    edges [0.0, 0.0, -0.0, 0.0]  mean dx -0.00  dy +0.00
  YOLO - vertex rect edges [0.0, 0.0, -0.0, 0.0]  mean dx -0.00  dy +0.00
image 3: box is 30.2% of the image; filename box vs vertex rect gap 0.0% of longer side
  YOLO - filename    edges [0.0, -0.0, 0.0, -0.0]  mean dx +0.00  dy -0.00
  YOLO - vertex rect edges [0.0, -0.0, 0.0, -0.0]  mean dx +0.00  dy -0.00
image 4: box is 32.0% of the image; filename box vs vertex rect gap 0.0% of longer side
  YOLO - filename    edges [0.0, 0.0, -0.0, 0.0]  mean dx -0.00  dy +0.00
  YOLO - vertex rect edges [0.

In [ ]:
# vertex plot: the 4 largest boxes, zoomed. Red = YOLO box, green dashed = rectangle around the 4 vertices, cyan = vertices.
fig2, axes2 = plt.subplots(1, len(top), figsize=(4 * len(top), 3.5), squeeze=False)
for ax, (i, p) in zip(axes2[0], enumerate(top, 1)):
    y, f, v = boxes_px(p)
    px, py = zip(*ccpd.parse_name(member_of[p.name]).vertices)
    ax.imshow(Image.open(p).convert('RGB'))
    ax.add_patch(plt.Rectangle((y[0], y[1]), y[2] - y[0], y[3] - y[1], fill=False, edgecolor='red', lw=1.5))
    ax.add_patch(plt.Rectangle((v[0], v[1]), v[2] - v[0], v[3] - v[1], fill=False, edgecolor='lime', lw=1.5, ls='--'))
    ax.plot(px + px[:1], py + py[:1], '.-', color='cyan', ms=6, lw=1)
    m = 0.15 * max(y[2] - y[0], y[3] - y[1])
    ax.set_xlim(y[0] - m, y[2] + m)
    ax.set_ylim(y[3] + m, y[1] - m)
    ax.set_title(f'image {i}', fontsize=9)
    ax.axis('off')
fig2.suptitle('4 largest boxes: red YOLO, green dashed vertex rectangle, cyan vertices')
fig2.tight_layout()

## Step 7: save the grid to Drive
PNG goes to the private `reports/ccpd_yolo` folder on Drive, for the presentation. It must not be copied into the repo.

In [12]:
os.makedirs(REPORTS, exist_ok=True)
out = f'{REPORTS}/ccpd_yolo_grid.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print('saved', out)

saved /content/drive/MyDrive/alpr/reports/ccpd_yolo/ccpd_yolo_grid.png


## Before committing this notebook
This notebook shows CCPD images, so its image outputs must not be committed. Text and table outputs stay. The two image cells (the grid in step 6 and the plot in step 6b) carry the cell tag `remove_output`.
1. Restart & Run All in VS Code, then save.
2. Check the tags survived the save: `grep -c '"remove_output"' notebooks/02_ccpd_yolo_check.ipynb` must print 2. If not, run `python scripts/tag_notebook_cells.py notebooks/02_ccpd_yolo_check.ipynb` and check again.
3. From the repo root, strip the images: `uv run --with nbconvert jupyter nbconvert --to notebook --inplace --TagRemovePreprocessor.enabled=True --TagRemovePreprocessor.remove_all_outputs_tags='["remove_output"]' notebooks/02_ccpd_yolo_check.ipynb`
4. Check `grep -c '"image/png"' notebooks/02_ccpd_yolo_check.ipynb` prints 0.
5. Read `git diff notebooks/02_ccpd_yolo_check.ipynb` for filenames or plate text, then `git add` and commit. Copy the counts from step 3 and 5 into PLAN.md by hand.
After stripping, re-running the notebook brings the images back: strip again before the next commit.